# Differential Abundance of Myeloid States Between Treatment Stages

Tests differential abundance of myeloid neighbourhoods (milopy, edgeR NB-GLM with
graph-weighted spatial FDR) for four pairwise comparisons, adjusting for smoking status:

- adjacent normal vs treatment-naive
- treatment-naive vs on-treatment
- treatment-naive vs TKI-resistant
- on-treatment vs TKI-resistant

logFC > 0 means the neighbourhood is enriched in the second stage of the pair.
Neighbourhoods are annotated at two levels (`cell_state`, `cell_state_v2`) and plotted as
neighbourhood graphs and miloR-style DA beeswarms, as in the malignant analysis (1.6a).

In [ ]:
# R + edgeR are required by milopy (called through rpy2)
# !apt-get install -y r-base r-base-dev
# !R -e "install.packages(c('BiocManager', 'statmod'), repos='http://cran.rstudio.com/')"
# !R -e "BiocManager::install('edgeR', ask=FALSE, update=FALSE)"
# !pip install --quiet scanpy rpy2 git+https://github.com/emdann/milopy.git@30646f538481151b6101b5e2f133858d2171000a
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid

In [ ]:
import os
import warnings

import importlib.util
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore", category=FutureWarning)

# milopy (last release, unmaintained) breaks on recent scipy/pandas: it indexes a sparse
# matrix with a pandas Series and writes floats into an int column. Idempotent source
# patch, applied before import; no-op on older environments where the lines differ.
_core = pathlib.Path(importlib.util.find_spec("milopy").submodule_search_locations[0]) / "core.py"
_src = _core.read_text()
for _old, _new in [
    ('nhood_ixs = adata.obs["nhood_ixs_refined"] == 1\n',
     'nhood_ixs = (adata.obs["nhood_ixs_refined"] == 1).to_numpy()\n'),
    ('adata.obs["nhood_kth_distance"] = 0\n',
     'adata.obs["nhood_kth_distance"] = 0.0\n'),
]:
    if _old in _src:
        _src = _src.replace(_old, _new)
        print("patched milopy.core:", _new.strip())
_core.write_text(_src)

import milopy.core as milo
import milopy.utils

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad"

milo_output_dir = "./Milo"
figures_output_dir = "./figures/milo"

recist_col = "recist_v3"
sample_col = "ID_2"                     # one sample per ID_2; Milo counts cells per nhood per sample

include_samples = [
    "TN001_AN", "TN002_AN", "TN003_AN", "TN004_AN", "TN005_AN", "TN006_AN", "TN007_AN", "TN008_AN", "TN009_AN",
    "TN001", "TN002", "TN003", "TN004", "TN005", "TN006", "TN007", "TN008", "TN009",
    "P009", "P012", "P001", "P003", "P005", "P011", "P002", "P006", "P008", "P013",
    "R001", "R003", "R006", "R009", "R007", "R005", "R010",
]

# covariate adjusted for in the design (~ covariate + Condition)
covariate = "smoking_status_grp"
drop_samples_with_label = "unknown"     # samples with this covariate value are dropped; None to keep as a level

# pairwise comparisons: (reference, test); logFC > 0 = enriched in test
comparisons = {
    "AN_vs_TN": ("0_adj_normal", "1_treatment_naive"),
    "TN_vs_OT": ("1_treatment_naive", "3_on_treatment"),
    "TN_vs_R": ("1_treatment_naive", "4_TKI_resistance"),
    "OT_vs_R": ("3_on_treatment", "4_TKI_resistance"),
}

# neighbourhood construction
use_rep = "X_scVI_n_latent_hvg_10"
n_neighbors = 10
prop = 0.1                              # fraction of cells sampled as nhood index candidates
seed = 0

# results
spatial_fdr_alpha = 0.1            # SpatialFDR threshold for colouring / counting significant nhoods
mixed_threshold = 0.5                   # nhoods whose majority label is < this fraction -> "Mixed"
nhood_graph_basis = "X_umap"            # layout for the nhood graph plot

# neighbourhood annotation levels: obs column -> display order (first at the bottom of the beeswarm)
annotation_orders = {
    "cell_state": [
        "Macrophage INHBAhigh", "Macrophage INHBAhigh_infiltrating", "Macrophage INHBAhigh_inflammatory",
        "Macrophage CD36high", "Macrophage STAB1high", "Monocyte_CD16high", "MDSC", "cDC2",
        "cDC1_activatedDC", "Myeloid_dividing", "Mast cell", "Mixed",
    ],
    "cell_state_v2": [
        "Macrophage INHBAhigh", "Macrophage MERTKhigh", "Monocyte_CD16high", "MDSC", "cDC2",
        "cDC1_activatedDC", "Myeloid_dividing", "Mast cell", "Mixed",
    ],
}

# DA beeswarm (miloR plotDAbeeswarm equivalent)
beeswarm_exclude_mixed = True
beeswarm_logfc_limits = (-9, 9)         # nhoods outside are not shown (count printed). None = fit data
beeswarm_single_figsize = (7, 5)
# ggplot2 scales::muted("blue") / "white" / scales::muted("red")
da_cmap = LinearSegmentedColormap.from_list("milo_da", ["#3A3A98", "#FFFFFF", "#832424"])

# marker dotplots on the analysed cells
marker_panels = {
    "immunosuppressive": {
        "IS": ["CD163", "MRC1", "CD274", "VTCN1", "SIRPA", "CD200R1", "TEK", "CCL22", "CCL17", "CCL18", "MERTK", "CD86"],
        "intrac": ["ARG1", "IDO1", "STAT3", "HMOX1"],
        "cyto": ["IL10", "TGFB1", "PTGES2"],
        "MDSC": ["CD14", "CCR2", "FUT4", "ITGAM", "CD33", "HLA-DRA", "HLA-DRB1", "HLA-DQA1"],
        "CD16- classical mono": ["FCGR3A", "ITGAX", "ITGAM", "FCGR1A", "CD52", "CD163"],
        "Receptor_costim": ["CD47", "SIRPA", "CSF1R", "CD40", "CD274", "PDCD1", "CD80", "CD86", "CCR2", "IL4R",
                            "CCR5", "IL12A", "MARCO", "ARG1", "CXCR4", "HAVCR2", "SIGLEC10", "IDO1", "LILRB1", "LILRB2"],
        "Chemokine_Cytokine": ["CCL5", "CCL20", "IL6", "TNF", "IL1B", "CCL22", "EGF", "IL10", "TGFB1", "CCL3",
                               "VEGFA", "MMP9", "EPHA4", "CAMP"],
        "TLR-STING": ["TLR7", "TLR9", "TLR8", "TMEM173"],
    },
    "selected_genes": {         # saved as dotplot_selected_genes.pdf
        "HLA": ["HLA-DRA", "HLA-DRB1", "HLA-DQA1"],
        "IS": ["CD163", "MRC1"],
        "set0": ["FABP4", "LSAMP", "CD36"],
        "set1": ["CD14", "ITGAM", "CD33", "VCAN", "STON2"],
        "set2": ["CCL18", "SELENOP", "HMOX1", "PLA2G7", "LGMN", "GPNMB"],
        "set3": ["F13A1", "SPP1"],
        "MDSC": ["CD14", "CCR2", "FUT4", "ITGAM", "CD33"],
    },
}

os.makedirs(milo_output_dir, exist_ok=True)
os.makedirs(figures_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir

## 1. Load myeloid cells

In [ ]:
adata = sc.read(input_path)
missing = sorted(set(include_samples) - set(adata.obs[sample_col].astype(str)))
assert not missing, f"include_samples not found in {sample_col}: {missing}"
adata = adata[adata.obs[sample_col].isin(include_samples)].copy()

if drop_samples_with_label is not None:
    dropped = sorted(adata.obs.loc[adata.obs[covariate].astype(str) == drop_samples_with_label, sample_col].unique())
    adata = adata[adata.obs[covariate].astype(str) != drop_samples_with_label].copy()
    print(f"dropped {len(dropped)} samples with {covariate} = {drop_samples_with_label}: {dropped}")

print(adata.n_obs, "cells,", adata.obs[sample_col].nunique(), "samples")
print(pd.crosstab(adata.obs["cell_state"], adata.obs[recist_col]))

### Marker dotplots on the analysed cells

Log-normalised copy for plotting only; Milo uses the scVI latent space and raw counts per sample.

In [ ]:
plot_data = adata.copy()
plot_data.X = plot_data.layers["raw_counts"].copy() if "raw_counts" in plot_data.layers else plot_data.X
plot_data.uns.pop("log1p", None)
sc.pp.normalize_total(plot_data, target_sum=1e4)
sc.pp.log1p(plot_data)

for name, panel in marker_panels.items():
    present = {k: [g for g in dict.fromkeys(v) if g in plot_data.var_names] for k, v in panel.items()}
    absent = {k: sorted(set(v) - set(present[k])) for k, v in panel.items() if set(v) - set(present[k])}
    if absent:
        print(f"{name}: not in data {absent}")
    sc.pl.dotplot(plot_data, {k: v for k, v in present.items() if v}, groupby="cell_state", use_raw=False,
                  standard_scale="var", cmap="Blues", dendrogram=True, dot_min=0.05 if name == "selected_genes" else None,
                  figsize=(7.5, 2.4) if name == "selected_genes" else (17, 3),
                  save=f"{name}.pdf" if name == "selected_genes" else None)
del plot_data

## 2. Design check

Milo fits one model per comparison at sample level, so the covariate × condition
table that matters is the number of **samples**, not cells. A covariate level present
in only one condition is partly confounded with it and weakens the adjustment.

In [ ]:
design_tables = {}
for name, (ref, test) in comparisons.items():
    obs = adata.obs[adata.obs[recist_col].isin([ref, test])]
    samples = obs.groupby(sample_col, observed=True)[[covariate, recist_col]].first()
    ambiguous = obs.groupby(sample_col, observed=True)[[covariate, recist_col]].nunique().max()
    assert (ambiguous <= 1).all(), f"{name}: a sample maps to more than one {covariate}/{recist_col} value"
    table = pd.crosstab(samples[covariate], samples[recist_col]).reindex(columns=[ref, test], fill_value=0)
    design_tables[name] = table
    print(f"--- {name}: samples per {covariate} x condition")
    print(table, "\n")
    if (table == 0).any(axis=None):
        print(f"WARNING {name}: some {covariate} levels are absent from one condition\n")

## 3. Run Milo per comparison

For each comparison: subset to the two stages, rebuild the kNN graph on the scVI
latent space, sample neighbourhoods, count cells per sample, and test
`~ covariate + Condition` (the Condition coefficient is tested; its reference level is
the first stage of the pair). Each neighbourhood is annotated at every level in
`annotation_orders`, stored as `annotation_<level>` (with `annotation_frac_<level>`).

In [ ]:
def run_milo(ad, ref, test):
    sub = ad[ad.obs[recist_col].isin([ref, test])].copy()
    sub.obs["Condition"] = pd.Categorical(sub.obs[recist_col].astype(str), categories=[ref, test])
    sub.obs[covariate] = sub.obs[covariate].astype(str).astype("category")

    sc.pp.neighbors(sub, use_rep=use_rep, n_neighbors=n_neighbors, key_added="milo", random_state=seed)
    np.random.seed(seed)
    milo.make_nhoods(sub, neighbors_key="milo", prop=prop)
    milo.count_nhoods(sub, sample_col=sample_col)
    milo.DA_nhoods(sub, design=f"~{covariate}+Condition")      # tests the last coefficient (Condition)

    res = sub.uns["nhood_adata"].obs
    for level, order in annotation_orders.items():
        milopy.utils.annotate_nhoods(sub, anno_col=level)
        label = res["nhood_annotation"].astype(str)
        label[res["nhood_annotation_frac"] < mixed_threshold] = "Mixed"
        res[f"annotation_{level}"] = pd.Categorical(label, categories=order)
        res[f"annotation_frac_{level}"] = res["nhood_annotation_frac"]
        unordered = sorted(set(label) - set(order))
        assert not unordered, f"labels missing from annotation_orders['{level}']: {unordered}"
    res.drop(columns=["nhood_annotation", "nhood_annotation_frac"], inplace=True)
    milopy.utils.build_nhood_graph(sub, basis=nhood_graph_basis)
    return sub

milo_results = {}
for name, (ref, test) in comparisons.items():
    print(f"=== {name}: {ref} (ref) vs {test}")
    sub = run_milo(adata, ref, test)
    res = sub.uns["nhood_adata"].obs
    res.to_csv(f"{milo_output_dir}/nhood_milo_myeloid_{name}_{covariate}_prop{prop}.csv")
    milo_results[name] = sub
    n_sig = (res["SpatialFDR"] < spatial_fdr_alpha).sum()
    print(f"{len(res)} nhoods, {n_sig} with SpatialFDR < {spatial_fdr_alpha} "
          f"({(res.loc[res.SpatialFDR < spatial_fdr_alpha, 'logFC'] > 0).sum()} up in {test})\n")

### QC: neighbourhood sizes, annotation purity, p-value distributions and volcano plots

In [ ]:
fig, axes = plt.subplots(4, len(comparisons), figsize=(4.5 * len(comparisons), 13))
for c, (name, sub) in enumerate(milo_results.items()):
    res = sub.uns["nhood_adata"].obs
    axes[0, c].hist(np.asarray(sub.obsm["nhoods"].sum(0)).ravel(), bins=100)
    axes[0, c].set(title=name, xlabel="Nhood size (cells)")
    for level in annotation_orders:
        axes[1, c].hist(res[f"annotation_frac_{level}"], bins=np.linspace(0, 1, 31), alpha=0.6, label=level)
    axes[1, c].axvline(mixed_threshold, ls="--", c="grey", lw=0.8)
    axes[1, c].set(xlabel="Majority-label fraction")
    axes[1, c].legend(fontsize=7, frameon=False)
    axes[2, c].hist(res["PValue"], bins=50)
    axes[2, c].set(xlabel="P-value")
    axes[3, c].scatter(res["logFC"], -np.log10(res["SpatialFDR"]), s=3)
    axes[3, c].axhline(-np.log10(spatial_fdr_alpha), ls="--", c="grey", lw=0.8)
    axes[3, c].set(xlabel="logFC", ylabel="-log10(SpatialFDR)")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/milo_myeloid_qc.pdf", bbox_inches="tight")
plt.show()

## 4. Neighbourhood graphs

Nodes are neighbourhoods at the UMAP position of their index cell, sized by number of
cells; colour is logFC for neighbourhoods with SpatialFDR < alpha (grey otherwise).
All panels share one symmetric colour scale and one alpha.

In [ ]:
vmax = max(sub.uns["nhood_adata"].obs.loc[sub.uns["nhood_adata"].obs.SpatialFDR < spatial_fdr_alpha, "logFC"].abs().max()
           for sub in milo_results.values())
vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1

fig, axes = plt.subplots(1, len(comparisons), figsize=(5.5 * len(comparisons), 5))
for ax, (name, sub) in zip(axes, milo_results.items()):
    nh = sub.uns["nhood_adata"]
    res, pos = nh.obs, nh.obsm["X_milo_graph"]
    sig = (res["SpatialFDR"] < spatial_fdr_alpha).values
    size = res["Nhood_size"].values * 0.5
    order = np.argsort(np.where(sig, res["logFC"].abs().values, -1))
    ax.scatter(pos[order][~sig[order], 0], pos[order][~sig[order], 1], s=size[order][~sig[order]],
               c="lightgrey", linewidths=0, rasterized=True)
    sc_ = ax.scatter(pos[order][sig[order], 0], pos[order][sig[order], 1], s=size[order][sig[order]],
                     c=res["logFC"].values[order][sig[order]], cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                     linewidths=0, rasterized=True)
    ref, test = comparisons[name]
    ax.set_title(f"{ref} → {test}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
fig.colorbar(sc_, ax=axes, label="logFC (SpatialFDR < %.2g)" % spatial_fdr_alpha, shrink=0.7)
plt.savefig(f"{figures_output_dir}/milo_myeloid_nhood_graph_{len(comparisons)}comparisons.pdf", dpi=200, bbox_inches="tight")
plt.show()

## 5. DA beeswarm by cell state

Same two-layer beeswarm as the malignant analysis: non-significant neighbourhoods
small, light grey and transparent underneath; significant ones (SpatialFDR < alpha)
coloured by logFC on top. One figure per comparison and annotation level, plus a
combined panel per level. All panels share one symmetric colour scale.

In [ ]:
def _van_der_corput(n, base=2):
    seq = np.zeros(n)
    for i in range(n):
        f, x, k = 1.0, 0.0, i + 1
        while k:
            f /= base
            x += f * (k % base)
            k //= base
        seq[i] = x
    return seq

def quasirandom_offsets(values, width):
    """Offsets in [-width, width] scaled by local density of `values`."""
    n = len(values)
    if n < 3 or np.std(values) == 0:
        return np.zeros(n)
    density = gaussian_kde(values)(values)
    density = density / density.max()
    order = np.argsort(values)
    offsets = np.empty(n)
    offsets[order] = (_van_der_corput(n) - 0.5) * 2
    return offsets * density * width

def plot_da_beeswarm(res, ax, title, vmax, level, alpha=spatial_fdr_alpha, limits=beeswarm_logfc_limits):
    res = res.copy()
    res["nhood_annotation"] = res[f"annotation_{level}"].astype(str)
    state_order = annotation_orders[level]
    if beeswarm_exclude_mixed:
        res = res[res["nhood_annotation"] != "Mixed"]
    if limits is not None:
        outside = ~res["logFC"].between(*limits)
        if outside.any():
            print(f"{title}: {outside.sum()} nhoods outside logFC {limits} not shown "
                  f"({(outside & (res['SpatialFDR'] < alpha)).sum()} significant)")
        res = res[~outside]
    groups = [s for s in state_order if s in set(res["nhood_annotation"])]
    pts = None
    for y, grp in enumerate(groups):
        d = res[res["nhood_annotation"] == grp]
        sig = (d["SpatialFDR"] < alpha).values
        ns_lfc, sig_lfc = d["logFC"].values[~sig], d["logFC"].values[sig]
        ax.scatter(ns_lfc, y + quasirandom_offsets(ns_lfc, 0.15), s=2, c="#CCCCCC", alpha=0.1,
                   linewidths=0, rasterized=True, zorder=1)
        pts = ax.scatter(sig_lfc, y + quasirandom_offsets(sig_lfc, 0.4), s=12, c=sig_lfc, cmap=da_cmap,
                         vmin=-vmax, vmax=vmax, linewidths=0, rasterized=True, zorder=2)
    ax.axvline(0, ls="--", color="black", lw=0.8, zorder=0)
    ax.set_yticks(range(len(groups)), groups, fontsize=10)
    ax.set_xlabel("Log Fold Change")
    ax.set_title(title, fontsize=9)
    if limits is not None:
        ax.set_xlim(*limits)
    return pts


all_res = pd.concat([sub.uns["nhood_adata"].obs.assign(comparison=name) for name, sub in milo_results.items()])
sig_lfc = all_res.loc[all_res["SpatialFDR"] < spatial_fdr_alpha, "logFC"].abs()
da_vmax = sig_lfc.max() if len(sig_lfc) else 1.0

for level in annotation_orders:
    for name, (ref, test) in comparisons.items():
        fig, ax = plt.subplots(figsize=beeswarm_single_figsize)
        pts = plot_da_beeswarm(all_res[all_res["comparison"] == name], ax, f"{ref} → {test}", da_vmax, level)
        ax.set_ylabel("Nhood annotation")
        if pts is not None:
            fig.colorbar(pts, ax=ax, label="logFC")
        plt.tight_layout()
        plt.savefig(f"{figures_output_dir}/nhood_milo_myeloid_{name}_{level}_beeswarm_FDR{spatial_fdr_alpha}_{covariate}.pdf",
                    dpi=200, bbox_inches="tight")
        plt.show()

    fig, axes = plt.subplots(1, len(comparisons), figsize=(5 * len(comparisons), 4.5), sharey=True)
    for ax, (name, (ref, test)) in zip(axes, comparisons.items()):
        pts = plot_da_beeswarm(all_res[all_res["comparison"] == name], ax, f"{ref} → {test}", da_vmax, level)
    axes[0].set_ylabel("Nhood annotation")
    fig.colorbar(pts, ax=axes, label=f"logFC (SpatialFDR < {spatial_fdr_alpha})", shrink=0.8)
    plt.savefig(f"{figures_output_dir}/milo_myeloid_DA_beeswarm_{level}_{len(comparisons)}comparisons.pdf",
                dpi=200, bbox_inches="tight")
    plt.show()

### Summary: significant neighbourhoods per state and direction

In [ ]:
for level in annotation_orders:
    summary = (all_res.assign(direction=np.where(all_res["logFC"] > 0, "up_in_test", "up_in_ref"),
                              significant=all_res["SpatialFDR"] < spatial_fdr_alpha)
               .groupby(["comparison", f"annotation_{level}", "direction"], observed=True)["significant"]
               .agg(n_nhoods="size", n_significant="sum")
               .reset_index())
    summary.to_csv(f"{milo_output_dir}/milo_myeloid_summary_{level}.csv", index=False)
    display(summary.pivot_table(index=f"annotation_{level}", columns=["comparison", "direction"],
                                values="n_significant", fill_value=0, observed=True))